In [ ]:
import os
WRI_PROJECT_ROOT = os.environ.get("WRI_PROJECT_ROOT", "/home/shares/wwri-wildfire")

# script to rescale WUI values to 0-1 range nased on our scoring matrix
# The data are organized in tiles of 100 km x 100 km and follow the EQUI7 tiling grid and projection system. 
# The images are compressed GeoTiff files (*.tif). There is a mosaic in GDAL Virtual format (*.vrt), which can readily be opened in 
# most Geographic Information Systems. Please consider the generation of image pyramids before using *.vrt files.
# The raster dataset contains Wildland-urban interface (WUI) data (one layer), 10 m spatial resolution, 8 discrete classes:
# intermix WUI (where buildings and wildland vegetation intermingle)
# interface WUI (where buildings are close to large wildland vegetation patches)
# 0 - Non-Vegetated / Non-WUI
# 1 - Forest/Shrubland/Wetland-dominated Intermix WUI
# 2 - Forest/Shrubland/Wetland-dominated Interface WUI
# 3 - Grassland-dominated Intermix WUI
# 4 - Grassland -dominated Interface WUI
# 5 - Non-WUI: Forest/Shrub/Wetland-dominated
# 6 - Non-WUI: Grassland-dominated
# 7 - Non-WUI: Urban
# 8 - Non-WUI: Other
# In addition, the data contain tabular data on WUI area, population and biomass in
# the WUI, as well as wildfire area and people affected by wildfire in the WUI per world region, country, subnational administrative unit and biome.
# The data also contain the key algorithm for WUI mapping (also accessible here: https://github.com/franzschug/global_wildland_urban_interface).

# script takes 85 mins to run
import os
import numpy as np
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt

# --- Paths ---
input_vrt = os.path.join(WRI_PROJECT_ROOT, 'data', 'infrastructure', 'intermediate', 'study_area_wui_map_5070_fixed.vrt')
output_path = os.path.join(WRI_PROJECT_ROOT, 'final_layers', '2024', 'infrastructure', 'indicators', 'infrastructure_resistance_wildland_urban_interface_unmasked.tif')

# --- Reclassification map ---
reclass_map = {
    0: 1,
    1: 0,
    2: 0.5,
    3: 0,
    4: 0.5,
    5: 1,
    6: 1,
    7: 1,
    8: 1
}

def reclassify_array(array, mapping):
    out = np.full(array.shape, np.nan, dtype='float32')
    for k, v in mapping.items():
        out[array == k] = v
    return out

# --- Read source and write to a proper GTiff ---
with rasterio.open(input_vrt) as src:
    print("✅ Opened VRT input raster.")
    
    # Fix the profile: use GTiff instead of VRT and allow large files
    profile = src.profile.copy()
    profile.update(
        driver='GTiff',
        dtype='float32',
        count=1,
        compress='lzw',
        nodata=np.nan,
        BIGTIFF='YES'  # <-- CRUCIAL FIX
    )

    print(f"📦 Creating output raster at: {output_path}")
    
    with rasterio.open(output_path, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            block = src.read(1, window=window)
            reclassified = reclassify_array(block, reclass_map)
            dst.write(reclassified, 1, window=window)

print("✅ Reclassification written successfully.")

#✅ Opened VRT input raster.
#📦 Creating output raster at: output_path (see variable above)
#✅ Reclassification written successfully.